In [0]:
# ============================================================

# GOLD – dim_client

# Grain: source_user_id (client / tenant)

# ============================================================

from pyspark.sql import functions as F

from pyspark.sql.window import Window

# ---------------- CONFIG ----------------

SILVER_TICKETS = "workspace.slainte_silver.easyvista_tickets_silver_demo"

GOLD_DB = "workspace.slainte_gold"

DIM_CLIENT = f"{GOLD_DB}.dim_client"

# ---------------- SETUP ----------------

spark.sql(f"CREATE DATABASE IF NOT EXISTS {GOLD_DB}")

df_tickets = spark.table(SILVER_TICKETS)

# ---------------- BUILD DIM ----------------

df_client_base = (

    df_tickets

    .select(F.col("source_user_id"))

    .filter(F.col("source_user_id").isNotNull())

    .distinct()

)

window = Window.orderBy("source_user_id")

df_dim_client = (

    df_client_base

    .withColumn("client_id", F.row_number().over(window))

    .withColumn("client_active", F.lit(True))

    .withColumn("created_at", F.current_timestamp())

    .select(

        "client_id",

        "source_user_id",

        "client_active"

    )

)

# ---------------- WRITE ----------------

df_dim_client.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(DIM_CLIENT)

print("✅ dim_client created successfully")

display(spark.table(DIM_CLIENT).orderBy("client_id"))
 